In [1]:
%pip install -e ..
%load_ext autoreload 
%autoreload 2
import numpy as np
import torch

import notebook_setup
from src.ccpfn import CEPOEstimator 

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Obtaining file:///layer6share/chris/Projects/CCPFN-inference/CCPFN-inference
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ccpfn (pyproject.toml) ... done
  Created wheel for ccpfn: filename=ccpfn-1.0.0-py3-none-any.whl size=5669 sha256=3d26a167be6df3b32f1678cf980c695bf846ca6b1d4aac89d80ad64df9d6b745
  Stored in directory: /tmp/pip-ephem-wheel-cache-sq4qhbi9/wheels/69/25/30/e67e178c45881d09b10d7c8bd49f53b05aadad1f16e8a9f07b
Successfully built ccpfn
  Attempting uninstall: ccpfn
    Found existing installation: ccpfn 1.0.0
    Uninstalling ccpfn-1.0.0:
      Successfully uninstalled ccpfn-1.0.0
Note: you may need to restart the kernel to use updated packages.


## Example: Running CCPFN

In this example, we demonstrate how to use CCPFN to predict individual treatment-response curves (ITRC). For simplicity, this notebook uses fully synthetic data. 

In [5]:
# Generate data

from src.ccpfn.benchmarks import ADMITDataset

meta_dataset = ADMITDataset(
    n_tables=1,
    test_ratio=0.2,
)
data = meta_dataset[0][0]

In [6]:
data

CEPO_Dataset(X_train=array([[0.0388737 , 0.65894279, 0.27255775, 0.87154568, 0.48499228,
        0.30557646],
       [0.84253451, 0.30320819, 0.4294479 , 0.47070595, 0.15737158,
        0.03106813],
       [0.7500341 , 0.44196536, 0.15097568, 0.80635098, 0.46070553,
        0.07531232],
       ...,
       [0.05136107, 0.60814294, 0.64205715, 0.66526292, 0.3718805 ,
        0.80594489],
       [0.26655012, 0.34487378, 0.64904447, 0.8828118 , 0.18413002,
        0.84036151],
       [0.74757464, 0.73054263, 0.04635327, 0.54966967, 0.46699145,
        0.92784227]], shape=(3276, 6)), t_train=array([0.85984183, 0.23016091, 0.2689302 , ..., 0.91024477, 0.92124156,
       0.17407799], shape=(3276,)), y_train=array([-0.6705663 ,  0.30996487, -0.30151656, ..., -2.04390361,
       -1.02268242, -0.39534039], shape=(3276,)), X_test=array([[0.88234761, 0.82855141, 0.65206582, 0.45841383, 0.93295604,
        0.83074655],
       [0.30299093, 0.09762824, 0.64445236, 0.18074031, 0.25740489,
        0.88

# Single CEPO Estimation

In [ ]:
estimator = CEPOEstimator(device=device)
estimator.fit(
    warfarin_dataset_cepo.X_train,
    warfarin_dataset_cepo.t_train,
    warfarin_dataset_cepo.y_train,
)

estimated_cepos = estimator.estimate_cepos(warfarin_dataset_cepo.X_test, warfarin_dataset_cepo.t_test)

# Dose Response Visualization

In [ ]:
import matplotlib.pyplot as plt

estimator = CEPOEstimator(device=device)
estimator.fit(
    warfarin_dataset_cepo.X_train,
    warfarin_dataset_cepo.t_train,
    warfarin_dataset_cepo.y_train,
)

# Choose one covariate set and evaluate its potential outcome over a grid of doses.
covariate_index = 0
x_fixed = warfarin_dataset_cepo.X_test[covariate_index : covariate_index + 1]

t_min = warfarin_dataset_cepo.t_train.min()
t_max = warfarin_dataset_cepo.t_train.max()
t_grid = np.linspace(t_min, t_max, 100)
X_grid = np.repeat(x_fixed, len(t_grid), axis=0)

estimated_response = estimator.estimate_cepo(X_grid, t_grid)

plt.figure(figsize=(7, 4))
plt.plot(t_grid, estimated_response, label="Estimated dose-response")
plt.scatter(
    warfarin_dataset_cepo.t_test[covariate_index],
    warfarin_dataset_cepo.true_cepo[covariate_index],
    color="black",
    zorder=3,
    label="Held-out CEPO target",
)
plt.xlabel("Dose")
plt.ylabel("Expected outcome")
plt.title(f"Dose-response for covariate set {covariate_index}")
plt.legend()
plt.tight_layout()
plt.show()